# Punto 1 — Clasificación para Focalización de Programas Sociales
**Taller 1 · HE2 · Consultoría Económica con IA Responsable**

**Cliente:** BID · **País:** Costa Rica · **Programa:** IMAS / SINIRUBE

Este notebook implementa el modelamiento de clasificación binaria a partir del dataset preprocesado
(`train_cleaned_hogar.csv`). La métrica de optimización es **F₂**, derivada de la estructura de
costos asimétricos del programa.

| Sección | Contenido |
|---|---|
| 1 | Carga de datos |
| 2 | Train / Test split |
| 3 | Análisis de costos — elección de métrica |
| 4 | Funciones auxiliares |
| 5 | Regresión Logística — modelo base |
| 6 | Tuning con GridSearchCV (F₂) |
| 7 | Evaluación del modelo tuneado |
| 8 | Análisis de umbral con costos reales BID |
| 9 | Exportación del modelo |

In [ ]:
import warnings
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    fbeta_score, precision_score, recall_score,
    roc_auc_score, make_scorer, ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Costos del programa BID (USD) ──────────────────────────────────────────
C_FN = 2_500 + 900 + 400   # $3 800: hogar pobre excluido del subsidio
C_FP = 1_200 + 150          # $1 350: hogar no-pobre que recibe el subsidio

BETA_TEORICO = (C_FN / C_FP) ** 0.5   # sqrt(3800/1350) ≈ 1.677
BETA         = 2                        # redondeo dentro de {0.5, 1, 2}

RANDOM_STATE = 42
TEST_SIZE    = 0.2
SCORING      = make_scorer(fbeta_score, beta=BETA, zero_division=0)

print('Configuración cargada:', {
    'C_FN': C_FN, 'C_FP': C_FP,
    'β_teórico': round(BETA_TEORICO, 3), 'β_usado': BETA,
    'RANDOM_STATE': RANDOM_STATE, 'TEST_SIZE': TEST_SIZE,
})

## 1. Carga de datos

El dataset fue generado por `preprocesamiento_punto1.py` y almacenado en `Datos/train_cleaned_hogar.csv`.
Ya incluye: imputación, limpieza, agregación al nivel de hogar, feature engineering (4 índices compuestos)
y estandarización de variables continuas. **Una fila = un hogar.**

In [ ]:
DATA_PATH = os.path.join('Datos', 'train_cleaned_hogar.csv')

df = pd.read_csv(DATA_PATH)

n0   = (df['Target'] == 0).sum()
n1   = (df['Target'] == 1).sum()
prop = df['Target'].mean()

print(f'Shape: {df.shape[0]:,} hogares x {df.shape[1]} variables')
print(f'Target — 0 (No pobre): {n0:,}  |  1 (Pobre): {n1:,}')
print(f'Proporcion de hogares pobres: {prop:.3f}')

display(df.head(3))

## 2. Train / Test split

Partición estratificada 80/20 para preservar el balance de clases en ambos conjuntos.

In [ ]:
X = df.drop(columns=['Target', 'idhogar'], errors='ignore')
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'Balance train: {y_train.mean():.3f}  |  Balance test: {y_test.mean():.3f}')
print(f'Features del modelo: {X_train.shape[1]}')

## 3. Análisis de costos — elección de métrica

La métrica de optimización se deriva de la estructura de costos asimétrica del programa:

- **Falso negativo (FN):** hogar pobre clasificado como no-pobre → excluido del subsidio.
- **Falso positivo (FP):** hogar no-pobre clasificado como pobre → recibe subsidio sin necesitarlo.

La fórmula para el β óptimo de F_β es: **β = √(C_FN / C_FP)**

In [ ]:
costos = pd.DataFrame([
    {'Error': 'FN', 'Concepto': 'Perdida de bienestar (ano sin asistencia)', 'USD': 2_500},
    {'Error': 'FN', 'Concepto': 'Costo de emergencia social posterior',      'USD':   900},
    {'Error': 'FN', 'Concepto': 'Costo politico / reputacional (por caso)',  'USD':   400},
    {'Error': 'FN', 'Concepto': '-- TOTAL FN --',                           'USD': C_FN},
    {'Error': 'FP', 'Concepto': 'Subsidio desperdiciado',                    'USD': 1_200},
    {'Error': 'FP', 'Concepto': 'Costo administrativo irrecuperable',        'USD':   150},
    {'Error': 'FP', 'Concepto': '-- TOTAL FP --',                           'USD': C_FP},
])
display(costos.set_index(['Error', 'Concepto']))

asimetria = C_FN / C_FP
print(f'Relacion C_FN / C_FP = {C_FN} / {C_FP} = {asimetria:.2f}')
print(f'Beta teorico  : beta = sqrt(C_FN/C_FP) = sqrt({asimetria:.2f}) = {BETA_TEORICO:.3f}')
print(f'Beta utilizado: beta = {BETA}  (en {{0.5, 1, 2}} segun pista del enunciado)')
print(f'F{BETA} penaliza los FN {BETA**2}x mas que los FP,')
print('alineado con el mandato del BID: proteger hogares vulnerables es la prioridad.')

## 4. Funciones auxiliares

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, beta=BETA):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'accuracy':       accuracy_score(y_true, y_pred),
        'precision':      precision_score(y_true, y_pred, zero_division=0),
        'recall':         recall_score(y_true, y_pred, zero_division=0),
        'f1':             f1_score(y_true, y_pred, zero_division=0),
        f'f{beta}':       fbeta_score(y_true, y_pred, beta=beta, zero_division=0),
        'roc_auc':        roc_auc_score(y_true, y_prob),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'cost_total_usd': C_FP * fp + C_FN * fn,
    }


def evaluate_thresholds(y_true, y_prob, thresholds, beta=BETA, c_fp=C_FP, c_fn=C_FN):
    rows = []
    for t in thresholds:
        y_pred_t = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_t).ravel()
        rows.append({
            'threshold':      t,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'precision':      precision_score(y_true, y_pred_t, zero_division=0),
            'recall':         recall_score(y_true, y_pred_t, zero_division=0),
            'f1':             f1_score(y_true, y_pred_t, zero_division=0),
            f'f{beta}':       fbeta_score(y_true, y_pred_t, beta=beta, zero_division=0),
            'cost_total_usd': c_fp * fp + c_fn * fn,
        })
    return pd.DataFrame(rows).sort_values('threshold').reset_index(drop=True)

## 5. Regresión Logística — modelo base

Modelo con hiperparámetros por defecto y umbral 0.5, para establecer una línea base.

In [ ]:
logit_base = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
logit_base.fit(X_train, y_train)

y_prob_base = logit_base.predict_proba(X_test)[:, 1]
y_pred_base = (y_prob_base >= 0.5).astype(int)

metrics_base = evaluate_model(y_test, y_pred_base, y_prob_base)

cols_show = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc',
             'tn', 'fp', 'fn', 'tp', 'cost_total_usd']
print('Metricas — Logit base (umbral 0.5):')
display(pd.DataFrame([metrics_base])[cols_show].round(4))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_base, ax=ax,
    display_labels=['No pobre', 'Pobre'],
    colorbar=False, cmap='Blues',
)
ax.set_title('Logit base — Matriz de confusion (umbral 0.5)')
plt.tight_layout()
plt.show()

## 6. Tuning con GridSearchCV (F₂)

`GridSearchCV` optimiza los hiperparámetros de la regresión logística maximizando F₂ en
validación cruzada estratificada de 5 folds.

**Hiperparámetros explorados:**
- `C`: inverso de la regularización. Valores bajos = más regularización.
- `penalty`: tipo de regularización (`l1` = Lasso, `l2` = Ridge).
- `class_weight`: `balanced` pondera las clases inversamente proporcional a su frecuencia,
  lo que aumenta el recall en la clase minoritaria (pobres) a costa de más FP.

In [ ]:
param_grid = {
    'C':            [0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty':      ['l1', 'l2'],
    'class_weight': [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2000,
        solver='liblinear',   # soporta tanto l1 como l2
        random_state=RANDOM_STATE,
    ),
    param_grid=param_grid,
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

grid.fit(X_train, y_train)

print(f'Mejores hiperparametros : {grid.best_params_}')
print(f'Mejor F{BETA} en CV      : {grid.best_score_:.4f}')

## 7. Evaluación del modelo tuneado

In [ ]:
best_logit   = grid.best_estimator_
y_prob_tuned = best_logit.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prob_tuned >= 0.5).astype(int)

metrics_tuned = evaluate_model(y_test, y_pred_tuned, y_prob_tuned)

# ── Comparacion base vs tuned ────────────────────────────────────────────
comp_cols = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
comp = pd.DataFrame([
    {'modelo': 'base',  **{k: metrics_base[k]  for k in comp_cols}},
    {'modelo': 'tuned', **{k: metrics_tuned[k] for k in comp_cols}},
]).set_index('modelo')

print('Comparacion base vs tuned (umbral 0.5):')
display(comp.round(4))

# ── Confusion matrix ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tuned, ax=ax,
    display_labels=['No pobre', 'Pobre'],
    colorbar=False, cmap='Blues',
)
ax.set_title(f'Logit tuneado — Confusion (umbral 0.5)')
plt.tight_layout()
plt.show()

## 8. Análisis de umbral con costos reales BID

El umbral por defecto (0.5) no es necesariamente óptimo. Se evalúan umbrales en [0.15, 0.80] para encontrar:
1. El umbral que **maximiza F₂** (métrica del concurso).
2. El umbral que **minimiza el costo total** según la estructura BID.

In [ ]:
thresholds = np.arange(0.15, 0.80, 0.05).round(2)
thresh_df  = evaluate_thresholds(y_test, y_prob_tuned, thresholds)

best_f2   = thresh_df.loc[thresh_df[f'f{BETA}'].idxmax()]
best_cost = thresh_df.loc[thresh_df['cost_total_usd'].idxmin()]

cols_thresh = ['threshold', 'fp', 'fn', 'precision', 'recall', f'f{BETA}', 'cost_total_usd']
print('Tabla umbral vs metricas / costo BID:')
display(thresh_df[cols_thresh].round(4))

_t_f2     = best_f2['threshold']
_score_f2 = best_f2[f'f{BETA}']
_cost_f2  = best_f2['cost_total_usd']
_t_cost   = best_cost['threshold']
_cost_min = best_cost['cost_total_usd']
_fp_min   = int(best_cost['fp'])
_fn_min   = int(best_cost['fn'])

print(f'Umbral que maximiza F{BETA}    : {_t_f2:.2f}  (F{BETA}={_score_f2:.4f}, Costo=${_cost_f2:,.0f})')
print(f'Umbral que minimiza Costo BID : {_t_cost:.2f}  (Costo=${_cost_min:,.0f}, FP={_fp_min}, FN={_fn_min})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: metricas vs umbral
axes[0].plot(thresh_df['threshold'], thresh_df['precision'],
             label='Precision', marker='o', ms=4)
axes[0].plot(thresh_df['threshold'], thresh_df['recall'],
             label='Recall',    marker='o', ms=4)
axes[0].plot(thresh_df['threshold'], thresh_df['f1'],
             label='F1',        marker='o', ms=4)
axes[0].plot(thresh_df['threshold'], thresh_df[f'f{BETA}'],
             label=f'F{BETA} (metrica objetivo)', marker='o', ms=4,
             linewidth=2.5, color='crimson')
axes[0].axvline(_t_f2, color='gray', linestyle='--',
                label=f'Optimo F{BETA} (t={_t_f2:.2f})')
axes[0].set_xlabel('Umbral')
axes[0].set_ylabel('Metrica')
axes[0].set_title('Metricas vs umbral de clasificacion')
axes[0].legend(fontsize=8)

# Panel derecho: costo total BID vs umbral
axes[1].plot(thresh_df['threshold'], thresh_df['cost_total_usd'],
             color='crimson', marker='o', ms=4, linewidth=2)
axes[1].axvline(_t_cost, color='gray', linestyle='--',
                label=f'Costo minimo (t={_t_cost:.2f})')
axes[1].set_xlabel('Umbral')
axes[1].set_ylabel('Costo total (USD)')
axes[1].set_title(f'Costo total BID vs umbral  (C_FN=${C_FN:,} | C_FP=${C_FP:,})')
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Analisis de umbral — Logit tuneado', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Exportación del modelo

Se exporta el modelo con el umbral que **maximiza F₂** (métrica del concurso).
El archivo incluye el modelo, el umbral, la lista de features y las métricas en test,
lo que permite reproducir predicciones sin re-entrenar.

In [ ]:
UMBRAL_FINAL = float(_t_f2)

export = {
    'modelo':        best_logit,
    'umbral':        UMBRAL_FINAL,
    'features':      list(X_train.columns),
    'beta':          BETA,
    'metricas_test': {k: round(v, 4)
                      for k, v in metrics_tuned.items()
                      if k not in ('tn', 'fp', 'fn', 'tp')},
}

model_path = 'Punto1_modelo.pkl'
joblib.dump(export, model_path)

_f2_test = metrics_tuned[f'f{BETA}']
print(f'Modelo exportado : {model_path}')
print(f'Umbral final     : {UMBRAL_FINAL}')
print(f'Features         : {len(X_train.columns)}')
print(f'F{BETA} en test   : {_f2_test:.4f}')